# 🧪 합성 데이터 생성 (sdg_hub 활용)

이 노트북은 **sdg_hub**를 사용하여 τ-Knowledge banking_knowledge 도메인의
학습 데이터를 합성 생성합니다.

## 생성 유형

| 유형 | 비율 | 설명 |
|------|------|------|
| Policy QA | ~30% | 정책 조건, 제한, 예외 관련 질답 |
| Policy Application | ~25% | 문서 + 고객 상황 → 허용/불허/확인 필요 |
| Tool Selection | ~20% | 현재 관찰 + 도구 → 다음 도구 호출 |
| Trajectory | ~15% | 대화 + 도구 관찰 → 행동 시퀀스 |
| Clarification | ~10% | 불완전 상황 → 후속 질문 |

## 프로파일
- **smoke**: 64 샘플 (빠른 검증용)
- **lab**: 2,000~5,000 샘플 (학습용)

> ⚠️ **교사 모델 엔드포인트가 필요합니다** (SDG_TEACHER_ENDPOINT)  
> 이 노트북은 학습자 경로가 아닌 **저작 경로**입니다.  
> 학습자는 이 과정 없이 준비된 번들을 직접 사용합니다.

In [ ]:
"""Load SDG config and check teacher endpoint."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_yaml_config, PROJECT_ROOT

load_env()

sdg_config = load_yaml_config("configs/sdg.yaml")

# Teacher endpoint check
teacher_endpoint = os.environ.get("SDG_TEACHER_ENDPOINT", "")
teacher_model = os.environ.get("SDG_TEACHER_MODEL", "")
teacher_api_key = os.environ.get("SDG_TEACHER_API_KEY", "")

print("=" * 70)
print("🔧 SDG 설정 및 교사 모델 확인")
print("=" * 70)
print(f"  파이프라인: {sdg_config['pipeline']['name']}")
print(f"  버전: {sdg_config['pipeline']['version']}")
print(f"  체크포인팅: {sdg_config['pipeline']['checkpointing']}")
print(f"  시드: {sdg_config['generation']['seed']}")
print()

print("--- 교사 모델 ---")
if teacher_endpoint:
    print(f"  ✅ 엔드포인트: {teacher_endpoint}")
    print(f"     모델: {teacher_model}")
    print(f"     API 키: {'✅ 설정됨' if teacher_api_key else '❌ 미설정'}")
    print(f"     최대 동시성: {sdg_config['teacher']['max_concurrent']}")
    print(f"     타임아웃: {sdg_config['teacher']['timeout_seconds']}초")
    print(f"     예산 한도: ${sdg_config['teacher']['budget_limit_usd']}")

    # Test connectivity
    try:
        import httpx
        resp = httpx.get(f"{teacher_endpoint}/models", timeout=10,
                        headers={"Authorization": f"Bearer {teacher_api_key}"} if teacher_api_key else {})
        if resp.status_code == 200:
            print(f"  ✅ 연결 성공")
        else:
            print(f"  ⚠️  HTTP {resp.status_code}")
    except Exception as exc:
        print(f"  ⚠️  연결 테스트 실패: {exc}")
else:
    print("  ❌ SDG_TEACHER_ENDPOINT가 설정되지 않았습니다.")
    print("     .env 파일에 교사 모델 엔드포인트를 설정하세요.")
    print("     합성 데이터 생성은 교사 모델이 필수입니다.")

# Show target counts
print("\n--- 목표 샘플 수 ---")
for profile_name in ["smoke", "lab"]:
    counts = sdg_config["generation"]["target_counts"].get(profile_name, {})
    print(f"  {profile_name}: {counts.get('total', '?')} 총")
    for stype, count in counts.items():
        if stype != "total":
            print(f"    {stype}: {count}")

In [ ]:
"""Generate smoke profile first (64 samples)."""

import subprocess
import sys

print("=" * 70)
print("🧪 Smoke 프로파일 생성 (64 샘플)")
print("=" * 70)

if not teacher_endpoint:
    print("❌ 교사 모델 엔드포인트 미설정 — 생성을 건너뜁니다.")
    print("   SDG_TEACHER_ENDPOINT를 .env에 설정하세요.")
else:
    generate_script = PROJECT_ROOT / "scripts" / "generate_synthetic.py"

    if generate_script.exists():
        cmd = [
            sys.executable, str(generate_script),
            "--config", "configs/sdg.yaml",
            "--profile", "smoke",
        ]
        print(f"  실행: {' '.join(cmd)}")
        print("  (교사 모델 호출이 진행됩니다...)\n")

        result = subprocess.run(
            cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT),
        )

        if result.returncode == 0:
            print("✅ Smoke 프로파일 생성 완료")
            print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        else:
            print(f"❌ 생성 실패 (exit code: {result.returncode})")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print("  ⚠️  generate_synthetic.py 스크립트를 찾을 수 없습니다.")
        print("  수동 실행:")
        print("    python scripts/generate_synthetic.py --config configs/sdg.yaml --profile smoke")

        # Try using sdg_hub directly
        print("\n  또는 sdg_hub API를 직접 사용:")
        print("    from sdg_hub import generate")
        print("    generate(config='configs/sdg.yaml', profile='smoke')")

In [ ]:
"""Run validation on generated data."""

print("=" * 70)
print("✅ 생성 데이터 검증")
print("=" * 70)

canonical_path = PROJECT_ROOT / sdg_config["output"]["canonical_path"]
rejected_path = PROJECT_ROOT / sdg_config["output"]["rejected_path"]

if canonical_path.exists():
    canonical_files = list(canonical_path.glob("*.jsonl"))
    print(f"  정규 데이터 파일: {len(canonical_files)}")

    total_accepted = 0
    for cf in canonical_files:
        with open(cf) as f:
            count = sum(1 for line in f if line.strip())
        total_accepted += count
        print(f"    {cf.name}: {count} 샘플")
    print(f"  총 수용: {total_accepted}")

    # Check rejected
    if rejected_path.exists():
        rejected_files = list(rejected_path.glob("*.jsonl"))
        total_rejected = 0
        for rf in rejected_files:
            with open(rf) as f:
                count = sum(1 for line in f if line.strip())
            total_rejected += count
        print(f"  총 거부: {total_rejected}")
        if total_accepted + total_rejected > 0:
            rate = total_accepted / (total_accepted + total_rejected) * 100
            print(f"  수용률: {rate:.1f}%")

    # Run validation script
    validate_script = PROJECT_ROOT / "scripts" / "validate_synthetic.py"
    if validate_script.exists():
        cmd = [
            sys.executable, str(validate_script),
            "--config", "configs/data-preparation.yaml",
        ]
        print(f"\n  검증 실행: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
        if result.returncode == 0:
            print("  ✅ 검증 통과")
            print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
        else:
            print(f"  ❌ 검증 실패")
            print(result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
else:
    print(f"  ⚠️  정규 데이터 경로 없음: {canonical_path}")
    print("     먼저 smoke 프로파일을 생성하세요.")

In [ ]:
"""Review quality report."""

import json

print("=" * 70)
print("📊 품질 리포트 검토")
print("=" * 70)

# Check validation config
val_config = sdg_config.get("validation", {})
print("검증 설정:")
print(f"  독립 검증기: {val_config.get('independent_validator', False)}")
print(f"  스키마 검증: {val_config.get('schema_validation', False)}")
print(f"  근거 확인: {val_config.get('grounding_check', False)}")
print(f"  조건 확인: {val_config.get('condition_check', False)}")
print(f"  도구 스키마 확인: {val_config.get('tool_schema_check', False)}")
print(f"  시뮬레이션 재생: {val_config.get('simulation_replay', False)}")
print(f"  중복 검사: {val_config.get('deduplication', {})}")
print(f"  오염 검사: {val_config.get('contamination_check', {})}")

# Look for quality reports
usage_path = PROJECT_ROOT / sdg_config["output"]["usage_accounting_path"]
if usage_path.exists():
    with open(usage_path) as f:
        usage = json.load(f)
    print(f"\n교사 모델 사용량:")
    for k, v in usage.items():
        print(f"  {k}: {v}")

logs_path = PROJECT_ROOT / sdg_config["output"]["logs_path"]
if logs_path.exists():
    log_files = list(logs_path.glob("*.log")) + list(logs_path.glob("*.jsonl"))
    print(f"\n로그 파일: {len(log_files)}")
    for lf in log_files[:5]:
        print(f"  {lf.name}")

print("\n  ℹ️  품질 검토 결과에 검증기 모델과 생성기 모델의 동일 여부를 기록하세요.")
print("     LLM 판정만으로는 완전한 검증이 되지 않습니다.")

In [ ]:
"""Generate lab profile (2000-5000 samples)."""

print("=" * 70)
print("🏭 Lab 프로파일 생성 (2,000~5,000 샘플)")
print("=" * 70)

lab_counts = sdg_config["generation"]["target_counts"].get("lab", {})
print(f"  목표 총 샘플: {lab_counts.get('total', '?')}")
for stype, count in lab_counts.items():
    if stype != "total":
        pct = count / lab_counts.get('total', 1) * 100
        print(f"    {stype}: {count} (~{pct:.0f}%)")

print("\n⚠️  이 목표는 수용된 샘플 수이며, 보장된 산출량이 아닙니다.")
print("   수용률에 따라 실제 생성 수는 더 많을 수 있습니다.")
print(f"   예산 한도: ${sdg_config['teacher']['budget_limit_usd']}")

if not teacher_endpoint:
    print("\n❌ 교사 모델 엔드포인트 미설정 — 생성을 건너뜁니다.")
else:
    generate_script = PROJECT_ROOT / "scripts" / "generate_synthetic.py"
    if generate_script.exists():
        cmd = [
            sys.executable, str(generate_script),
            "--config", "configs/sdg.yaml",
            "--profile", "lab",
        ]
        print(f"\n  실행 명령: {' '.join(cmd)}")
        print("  (이 과정은 상당한 시간이 소요됩니다)")
        print("  체크포인팅이 활성화되어 중단 후 재개 가능합니다.")

        # Optionally run (commented out for safety)
        # result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
        print("\n  ℹ️  안전을 위해 자동 실행하지 않습니다.")
        print("     위 명령어를 터미널에서 직접 실행하세요.")

In [ ]:
"""Final quality check."""

print("=" * 70)
print("✅ 최종 품질 확인")
print("=" * 70)

# Quality gates check
prep_config_loaded = load_yaml_config("configs/data-preparation.yaml")
quality_gates = prep_config_loaded.get("quality_gates", {})

print("품질 게이트 기준:")
print(f"  최소 수용률: {quality_gates.get('min_acceptance_rate', 0.7):.0%}")
print(f"  필수 유형: {quality_gates.get('required_types', [])}")
print(f"  최소 도구 예제: {quality_gates.get('min_tool_examples', 50)}")
print(f"  최소 궤적 예제: {quality_gates.get('min_trajectory_examples', 30)}")

if canonical_path.exists():
    print("\n현재 상태:")
    # Re-count
    import json as json_mod
    type_counts = {}
    total = 0
    for cf in canonical_path.glob("*.jsonl"):
        with open(cf) as f:
            for line in f:
                if line.strip():
                    try:
                        rec = json_mod.loads(line)
                        stype = rec.get("sample_type", "unknown")
                        type_counts[stype] = type_counts.get(stype, 0) + 1
                        total += 1
                    except Exception:
                        pass

    print(f"  총 수용 샘플: {total}")
    for stype, count in sorted(type_counts.items()):
        print(f"    {stype}: {count}")

    # Check gates
    tool_count = type_counts.get("tool_selection", 0) + type_counts.get("trajectory", 0)
    traj_count = type_counts.get("trajectory", 0)

    gate_results = [
        ("최소 도구 예제", tool_count >= quality_gates.get("min_tool_examples", 50)),
        ("최소 궤적 예제", traj_count >= quality_gates.get("min_trajectory_examples", 30)),
    ]

    for name, passed in gate_results:
        print(f"  {'✅' if passed else '❌'} {name}")
else:
    print("  ⚠️  생성 데이터가 없습니다.")

print("\n다음 단계:")
print("  📓 03_validate_and_release.ipynb — 검증 및 번들 릴리스")